<font size=7><strong>RECEIPT PARSING</strong></font> <a id="title"></a>

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Project**: *SmartBites*

- Beatriz Marques 20231605
- Constança Pereira da Silva 20231720
- Maria Inês Santos 20231630
- Mariana Calais-Pedro 20231641

*« notebook description »*

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [0. Imports](#0)
- [1. Load Sample Documents](#1)
- [2. Parse PDF](#2)
- [3. Parse Image](#3)

# <font color='#BFD72F' size=6>**0. Imports**</font> <a class="anchor" id="0"></a>

[Back to TOC](#toc)

In [30]:
# !pip install PyMuPDF
# !pip install frontend
# !pip install pytesseract
# !pip install filetype

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import sys
import os
import pandas as pd
import json

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../src')
calls_path = os.path.abspath('../calls')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from services.receipt_service import ReceiptService
from services.document_service import DocumentService

# Add the calls folder to sys.path
if calls_path not in sys.path:
    sys.path.append(calls_path)

import supabase_config

# ✅ CORRECT: Use the supabase client that's already created in supabase_config
supabase = supabase_config.supabase

print("✅ Supabase client loaded successfully!")

✅ Supabase client created successfully!
🔄 Initializing database tables...
Note: Tables might already exist or there was an error: {'message': "Could not find the table 'public.users' in the schema cache", 'code': 'PGRST205', 'hint': None, 'details': None}
✅ Supabase client loaded successfully!


# <font color='#BFD72F' size=6>**1. Load Sample Documents**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [2]:
# Load sample files
#pdf_path = "../data/sample_documents/sample_receipt.pdf"
#image_path = "../data/sample_documents/sample_receipt.jpg"

pdf_path = "../data/sample_documents/02_CONTINENTE_ExportedFromApp.pdf"
image_path = "../data/sample_documents/01_CONTINENTE_ExportedFromApp.jpg"

with open(pdf_path, "rb") as f:
    pdf_bytes = f.read()

with open(image_path, "rb") as f:
    image_bytes = f.read()

In [3]:
# Initialize services
receipt_service = ReceiptService()
document_service = DocumentService()

# <font color='#BFD72F' size=6>**2. Parse PDF**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

In [4]:
# Run receipt validation and analysis
try:
    print("Analyzing receipt...")
    receipt_data = receipt_service.analyze_receipt(pdf_bytes, mime_type="application/pdf")  # or "image/jpeg"
    print("Receipt Data:")
    print(json.dumps(receipt_data, indent=2, ensure_ascii=False))

    # Optional: Ask a question about the receipt
    question = "What was the total amount paid and how was it paid?"
    answer = receipt_service.answer_question(question, receipt_data)
    print("\nAnswer to question:")
    print(answer)

except Exception as e:
    print(f"Error during receipt analysis: {e}")

Analyzing receipt...
Receipt Data:
{
  "store_name": "CONTINENTE",
  "purchase_date": "2025-10-26",
  "purchase_time": "10:52",
  "invoice_number": "FS INQ002/297926",
  "items": [
    {
      "name": "DOURADA MEDIA (200-600G)",
      "section": "Peixaria",
      "quantity": 1,
      "unit_price": 7.06,
      "total_price": 7.06
    },
    {
      "name": "MEL DE FLORES TOP DOWN CNT 500G",
      "section": "Pequeno Almoco",
      "quantity": 1,
      "unit_price": 3.09,
      "total_price": 3.09
    },
    {
      "name": "BANANA RODELAS 200G CNT",
      "section": "Frutas e Legumes",
      "quantity": 3,
      "unit_price": 2.29,
      "total_price": 6.87
    },
    {
      "name": "BATATA CNT EMB.500GR",
      "section": "Frutas e Legumes",
      "quantity": 2,
      "unit_price": 2.19,
      "total_price": 4.38
    },
    {
      "name": "COG BRANCO LAMINADO CNT 250G",
      "section": "Frutas e Legumes",
      "quantity": 1,
      "unit_price": 1.59,
      "total_price": 1.59
    }

In [ ]:
'''
def insert_receipt_into_pantry(user_id: str, receipt_json: dict):
    """
    Takes JSON from OCR-extracted receipt and inserts items into pantry_items table.
    """

    store_name = receipt_json.get("store_name")
    purchase_date = receipt_json.get("purchase_date")
    purchase_time = receipt_json.get("purchase_time")
    items = receipt_json.get("items", [])

    rows_to_insert = []

    for item in items:
        rows_to_insert.append({
            "user_id": user_id,
            "item_name": item.get("name"),
            "quantity": item.get("quantity", 1),
            "store_name": store_name,
            "purchase_date": purchase_date,
            "purchase_time": purchase_time,
        })

    # Insert all items at once
    response = supabase.table("pantry_items").insert(rows_to_insert).execute()

    return response'''

In [12]:
insert_receipt_into_pantry(user_id = '5ad8f7c5-3beb-4489-9865-5083b930b71f', receipt_json = receipt_data)

APIResponse(data=[{'id': 2, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'DOURADA MEDIA (200-600G)', 'quantity': 1, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-26', 'purchase_time': '10:52', 'created_at': '2025-11-27T16:17:30.246361'}, {'id': 3, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'MEL DE FLORES TOP DOWN CNT 500G', 'quantity': 1, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-26', 'purchase_time': '10:52', 'created_at': '2025-11-27T16:17:30.246361'}, {'id': 4, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'BANANA RODELAS 200G CNT', 'quantity': 3, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-26', 'purchase_time': '10:52', 'created_at': '2025-11-27T16:17:30.246361'}, {'id': 5, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'BATATA CNT EMB.500GR', 'quantity': 2, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-26', 'purchase_time': '10:52', 'created_at': '2025-11-27T16:17:30.24

In [ ]:
#insert_one(receipts_collection, receipt_data)

InsertOneResult(ObjectId('691517360b6a7952291bfb6a'), acknowledged=True)

# <font color='#BFD72F' size=6>**3. Parse Image**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

In [14]:
try:
    print("Analyzing receipt image...")
    receipt_data = receipt_service.analyze_receipt(image_bytes, mime_type="image/jpeg")
    
    print("Receipt Data:")
    print(json.dumps(receipt_data, indent=2, ensure_ascii=False))
    
    question = "What items were purchased and what was the total?"
    answer = receipt_service.answer_question(question, receipt_data)

    print("\nAnswer to question:")
    print(answer)

except Exception as e:
    print(f"Error during receipt image analysis: {e}")

Analyzing receipt image...
Receipt Data:
{
  "store_name": "CONTINENTE",
  "purchase_date": "2025-10-13",
  "purchase_time": "17:12:43",
  "invoice_number": "EBZ001/1052334",
  "items": [
    {
      "name": "MOLHO DE SOJA CONTINENTE",
      "section": "Mercearia Salgada",
      "quantity": 1,
      "unit_price": 0.99,
      "total_price": 0.99
    },
    {
      "name": "BOL FRUIT&FORM FR BOSQUE 218G",
      "section": "Mercearia Doce",
      "quantity": 1,
      "unit_price": 1.99,
      "total_price": 1.99
    },
    {
      "name": "GOMAS AMORAS CONTINENTE L 300GR",
      "section": "Mercearia Doce",
      "quantity": 1,
      "unit_price": 1.89,
      "total_price": 1.89
    },
    {
      "name": "ICED TEA CONTINENTE MANGA 2L",
      "section": "Soft Drinks",
      "quantity": 1,
      "unit_price": 1.19,
      "total_price": 1.19
    },
    {
      "name": "GINGER ALE SCHWEPPES LATA 33CL",
      "section": "Soft Drinks",
      "quantity": 8,
      "unit_price": 0.72,
      "tota

In [15]:
insert_receipt_into_pantry(user_id = '5ad8f7c5-3beb-4489-9865-5083b930b71f', receipt_json = receipt_data)

APIResponse(data=[{'id': 9, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'MOLHO DE SOJA CONTINENTE', 'quantity': 1, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-13', 'purchase_time': '17:12:43', 'created_at': '2025-11-27T16:18:22.89197'}, {'id': 10, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'BOL FRUIT&FORM FR BOSQUE 218G', 'quantity': 1, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-13', 'purchase_time': '17:12:43', 'created_at': '2025-11-27T16:18:22.89197'}, {'id': 11, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'GOMAS AMORAS CONTINENTE L 300GR', 'quantity': 1, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-13', 'purchase_time': '17:12:43', 'created_at': '2025-11-27T16:18:22.89197'}, {'id': 12, 'user_id': '5ad8f7c5-3beb-4489-9865-5083b930b71f', 'item_name': 'ICED TEA CONTINENTE MANGA 2L', 'quantity': 1, 'store_name': 'CONTINENTE', 'purchase_date': '2025-10-13', 'purchase_time': '17:12:43', 'created_at

In [ ]:
"""-- Users table
create table if not exists public.users (
  id uuid primary key default gen_random_uuid(),
  username text unique,
  email text unique,
  full_name text,
  birth_date date,
  gender text,
  household_number integer,
  allergies jsonb,
  intolerances jsonb,
  restrictions jsonb,
  diet_type text,
  favorite_recipes jsonb,
  created_at timestamptz default now(),
  updated_at timestamptz default now()
);

-- Receipts table
create table if not exists public.receipts (
  id uuid primary key default gen_random_uuid(),
  user_id uuid references public.users(id) on delete cascade,
  store_name text,
  purchase_date date,
  purchase_time time,
  invoice_number text,
  items jsonb,
  subtotal numeric,
  discounts numeric,
  total numeric,
  payment_method text,
  raw_ocr_text text,
  parsed boolean default false,
  parsing_confidence numeric,
  created_at timestamptz default now(),
  updated_at timestamptz default now()
);

-- Pantry items table
create table if not exists public.pantry_items (
  id uuid primary key default gen_random_uuid(),
  user_id uuid references public.users(id) on delete cascade,
  name text,
  normalized_name text,
  quantity numeric,
  unit text,
  purchase_date date,
  source_receipt_id uuid references public.receipts(id),
  expiration_date date,
  category text,
  created_at timestamptz default now(),
  updated_at timestamptz default now()
);

create index if not exists idx_pantry_user_normalized on public.pantry_items (user_id, normalized_name);

-- Shopping list items
create table if not exists public.shopping_list_items (
  id uuid primary key default gen_random_uuid(),
  user_id uuid references public.users(id) on delete cascade,
  name text,
  normalized_name text,
  quantity numeric,
  unit text,
  section text,
  auto_added_by text,
  created_at timestamptz default now(),
  updated_at timestamptz default now()
);

create index if not exists idx_shopping_user_normalized on public.shopping_list_items (user_id, normalized_name);"""